# Practice Lab: Coding Agents (Lesson 11.7)

Module 11 . 8 exercises . CLAUDE.md + edit-test-debug + subagents + hooks + routing + sandbox + benchmark + PR

Exercises 1-5 run locally (pure Python simulations).


# Lesson 11.7: Coding Agents

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): CLAUDE.md for Your Project



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
claude_md = """# CLAUDE.md - Netsetos GenAI Course
## WHY - Course content repo, dark theme Wix-embedded HTML
## WHAT - /lessons/ HTML, /notebooks/ .ipynb, /solutions/ .py
## HOW
### Build: No build. HTML standalone. Notebooks: Colab run all.
### Test: jupyter nbconvert --execute *.ipynb | python -m pytest solutions/
### Lint: ruff check . && ruff format --check .
### Conventions: emerald #059669 (M9), DM Sans, Indian context, no em dashes
### Key Files: lessons/9.1.html (reference), notebooks/exercises-11_1.ipynb"""

lines = claude_md.strip().split("\n")
print(f"CLAUDE.md: {len(lines)} lines (target: <200)")
print("Structure: WHY -> WHAT -> HOW (build/test/lint/conventions/files)")
print("ETH Zurich: LLM-generated context REDUCED success ~3%")
print("Include ONLY what agents cannot discover independently")


</details>


---
## Exercise 2 (Easy): Edit-Test-Debug Loop



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

class ETDAgent:
    def __init__(self, retries=3): self.retries=retries
    def generate(self, task, error=""):
        if "fibonacci" in task.lower():
            if not error: return "def fib(n):\n    if n<=1: return n\n    return fib(n-1)+fib(n-2)\nassert fib(10)==55"
            return "def fib(n):\n    a,b=0,1\n    for _ in range(n): a,b=b,a+b\n    return a\nassert fib(10)==55"
        return "print('ok')"
    def execute(self, code):
        if "assert" in code:
            if np.random.random()<0.80: return {"pass":True,"out":"PASS"}
            return {"pass":False,"err":"RecursionError"}
        return {"pass":True,"out":"OK"}
    def solve(self, task):
        err=""
        for a in range(1,self.retries+1):
            code=self.generate(task,err); r=self.execute(code)
            print(f"    Attempt {a}: {'PASS' if r['pass'] else 'FAIL'}")
            if r["pass"]: return {"attempts":a,"passed":True}
            err=r.get("err",""); print(f"    Error: {err} -> fix")
        return {"attempts":self.retries,"passed":False}

print("Edit-Test-Debug Loop:")
for t in ["Write fibonacci, test fib(10)==55","Write palindrome checker","Sort dicts by age"]:
    print(f"  Task: '{t[:35]}...'"); ETDAgent().solve(t)
print("\n  Pattern: error=observation, fix=action, tests=termination")


</details>


---
## Exercise 3 (Medium): Subagent Orchestration



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SubAgent:
    def __init__(self,name,model,tools,ro=False):
        self.name=name;self.model=model;self.tools=tools;self.ro=ro
        self.cost={"haiku":0.001,"sonnet":0.01,"opus":0.05}.get(model,0.01)
    def run(self,prompt):
        used=[t for t in self.tools if not (self.ro and t in ["write","edit"])]
        return {"agent":self.name,"tools":used,"cost":self.cost}

class Parent:
    def __init__(self):
        self.explore=SubAgent("Explore","haiku",["read","glob","grep"],ro=True)
        self.general=SubAgent("General","sonnet",["read","write","edit","bash"])
    def solve(self,task):
        print(f"  Step 1: Explore (Haiku, read-only)")
        e=self.explore.run(task); print(f"    Tools: {e['tools']}, cost: ${e['cost']:.3f}")
        print(f"  Step 2: General (Sonnet, full tools)")
        g=self.general.run(task); print(f"    Tools: {g['tools']}, cost: ${g['cost']:.3f}")
        print(f"  Total: ${e['cost']+g['cost']:.3f}")
        print(f"\n  Depth=1: General cannot spawn sub-subagents")

print("Subagent Orchestration:")
Parent().solve("Add error handling to auth module")


</details>


---
## Exercise 4 (Medium): Hook-Based CI/CD



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class Hooks:
    def __init__(self): self.pre=[]; self.post=[]; self.log=[]
    def add_pre(self,name,fn): self.pre.append({"name":name,"fn":fn})
    def add_post(self,name,fn): self.post.append({"name":name,"fn":fn})
    def execute(self,tool,args):
        for h in self.pre:
            r=h["fn"](tool,args)
            if r.get("blocked"): self.log.append(f"BLOCKED:{tool}({args.get('path','')})"); return r
        self.log.append(f"OK:{tool}({args.get('path','')})")
        for h in self.post: h["fn"](tool,args,"ok")
        return {"blocked":False}

hooks=Hooks()
lint_runs=[]
hooks.add_pre("block_env",lambda t,a:{"blocked":True,"reason":f"Cannot modify {a.get('path','')}"} if t in ["write","edit"] and ".env" in a.get("path","") else {"blocked":False})
hooks.add_post("auto_lint",lambda t,a,o: lint_runs.append(f"ruff {a.get('path','')}") if t in ["write","edit"] and a.get("path","").endswith(".py") else None)

print("Hook-Based CI/CD:")
for tool,args in [("write",{"path":"src/main.py"}),("write",{"path":".env"}),("edit",{"path":"utils.py"}),("edit",{"path":".env.local"}),("read",{"path":"src/main.py"})]:
    r=hooks.execute(tool,args); print(f"  {('BLOCKED' if r.get('blocked') else 'OK'):>7}: {tool}({args['path']})")
print(f"\n  Auto-lint: {len(lint_runs)}x | Patterns: PreToolUse(block), PostToolUse(lint), SessionStart(load)")


</details>


---
## Exercise 5 (Medium): Model Routing



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Router:
    MODELS={"haiku":{"in":0.25,"out":1.25,"tasks":["read","grep","glob","ls","format","simple_edit"]},
            "sonnet":{"in":3.0,"out":15.0,"tasks":["implement","debug","test","refactor","review"]},
            "opus":{"in":15.0,"out":75.0,"tasks":["architecture","design","complex"]}}
    def route(self,tt):
        for m,c in self.MODELS.items():
            if tt in c["tasks"]: return m
        return "sonnet"
    def cost(self,m,i=1000,o=500): c=self.MODELS[m]; return (i*c["in"]+o*c["out"])/1e6

r=Router()
tasks=[("Read main.py","read",500,100),("Grep TODO","grep",300,200),("Format utils","format",800,600),
       ("Implement auth","implement",2000,1500),("Debug login","debug",3000,1000),("Write tests","test",2000,2000),
       ("Refactor DB","refactor",4000,3000),("Design API","architecture",3000,2000),("Fix typo","simple_edit",500,200),("List files","ls",200,100)]

print("Model Routing:")
tr=ts=0
for name,tt,i,o in tasks:
    m=r.route(tt); rc=r.cost(m,i,o); sc=r.cost("sonnet",i,o); tr+=rc; ts+=sc
    print(f"  {name:<18} {m:>8} ${rc:.5f} vs ${sc:.5f}")
print(f"\n  Routed: ${tr:.5f} vs Sonnet: ${ts:.5f} = {(1-tr/ts)*100:.0f}% savings")
print(f"  + Prompt caching: ${tr*(1-0.81):.5f} = {(1-tr*(1-0.81)/ts)*100:.0f}% total savings")


</details>


---
## Exercise 6 (Challenge): Kernel Sandbox
Design/architecture. See practice-lab-11_7.html.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Kernel Sandbox: Bubblewrap(namespace) + seccomp(syscall) + Landlock(filesystem)")
print("Prevents: process escape, kernel exploits, file access to /.ssh /etc/shadow")
print("Real escapes: rm -rf ~/ (fix: Landlock deny ~/), /proc escape (fix: seccomp block ptrace)")
print("Comparison: Kernel(low overhead,Codex) > Docker(medium) > MicroVM(best isolation,Firecracker)")


</details>


---
## Exercise 7 (Challenge): Internal Benchmark
Design/architecture. See practice-lab-11_7.html.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Internal Benchmark: 20 tasks from Netsetos codebase")
print("  10 bug fixes (checkQuiz, copy button, numbering, colors, progress bar)")
print("  5 features (expand-all, clipboard, responsive, theme toggle, progress)")
print("  5 refactors (extract CSS, convert inline, dedup JS, CSS vars, templates)")
print("Score: pass_rate x quality / normalized_cost")
print("Internal > SWE-bench: YOUR code, YOUR languages, no contamination")


</details>


---
## Exercise 8 (Challenge): Agentic PR Workflow
Design/architecture. See practice-lab-11_7.html.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Agentic PR Workflow: issue -> branch -> explore -> plan -> implement -> test -> PR")
print("  1. Issue labeled agent-ok, assigned to bot")
print("  2. git checkout -b agent/issue-123")
print("  3. Explore subagent (Haiku, read-only)")
print("  4-5. Plan + implement (Sonnet)")
print("  6-7. Run tests + self-review")
print("  8. gh pr create --draft")
print("  9-10. Auto-review (CodeRabbit) + human approval")
print("Permissions: contents:write + pull-requests:write ONLY")
print("Commits: Co-authored-by: Sarthak Kumar <sart@netsetos.com>")


</details>
